In [1]:
from dotenv import load_dotenv
load_dotenv()

True

In [2]:
import praw
import os
import json
from datetime import datetime

In [3]:
# Initialize Reddit API client
reddit = praw.Reddit(
    client_id=os.getenv('REDDIT_CLIENT_ID'),
    client_secret=os.getenv('REDDIT_SECRET'),
    user_agent='idea_finder_agent:v1.0 (by u/your_username)',  # Change this to your username
)

print(f"Reddit instance created. Read-only mode: {reddit.read_only}")
print(f"Client ID: {os.getenv('REDDIT_CLIENT_ID')[:10]}...")  # Show first 10 chars for verification

Reddit instance created. Read-only mode: True
Client ID: 7RqfB6RqZG...


In [4]:
def search_reddit_posts(query, subreddit_name="all", sort="relevance", time_filter="all", limit=25):
    """
    Search Reddit posts using PRAW
    
    Args:
        query (str): Search query (e.g., "saas ideas")
        subreddit_name (str): Subreddit to search in (default: "all")
        sort (str): Sort method - "relevance", "hot", "top", "new", "comments"
        time_filter (str): Time filter - "all", "day", "hour", "month", "week", "year"
        limit (int): Number of results to return (default: 25)
    
    Returns:
        list: List of dictionaries containing post information
    """
    try:
        # Get the subreddit
        subreddit = reddit.subreddit(subreddit_name)
        
        # Search for posts
        search_results = subreddit.search(
            query=query,
            sort=sort,
            time_filter=time_filter,
            limit=limit
        )
        
        posts = []
        for submission in search_results:
            post_data = {
                'title': submission.title,
                'author': str(submission.author) if submission.author else '[deleted]',
                'score': submission.score,
                'upvote_ratio': submission.upvote_ratio,
                'num_comments': submission.num_comments,
                'created_utc': datetime.fromtimestamp(submission.created_utc),
                'subreddit': str(submission.subreddit),
                'url': submission.url,
                'permalink': f"https://reddit.com{submission.permalink}",
                'selftext': submission.selftext[:200] + "..." if len(submission.selftext) > 200 else submission.selftext,
                'is_self': submission.is_self,
                'id': submission.id
            }
            posts.append(post_data)
        
        return posts
    
    except Exception as e:
        print(f"Error searching Reddit: {e}")
        return []

In [5]:
# Example: Search for SaaS ideas
print("Searching for 'saas ideas' on Reddit...")
saas_posts = search_reddit_posts(
    query="saas ideas",
    subreddit_name="entrepreneur+startups+SaaS",  # Search multiple relevant subreddits
    sort="relevance",
    time_filter="month",  # Posts from the last month
    limit=10
)

print(f"Found {len(saas_posts)} posts")
print("\n" + "="*80)

# Display results
for i, post in enumerate(saas_posts, 1):
    print(f"\n{i}. {post['title']}")
    print(f"   Author: u/{post['author']} | Subreddit: r/{post['subreddit']}")
    print(f"   Score: {post['score']} | Comments: {post['num_comments']} | Date: {post['created_utc'].strftime('%Y-%m-%d')}")
    if post['selftext']:
        print(f"   Text: {post['selftext']}")
    print(f"   Link: {post['permalink']}")
    print("-" * 60)

Searching for 'saas ideas' on Reddit...
Found 10 posts


1. SaaS Ideas?
   Author: u/I-Gone-Mad | Subreddit: r/SaaS
   Score: 11 | Comments: 31 | Date: 2025-08-31
   Text: How to find good ideas for a Web based SaaS ?

ANY RECOMMENDATIONS ? 
   Link: https://reddit.com/r/SaaS/comments/1n5chir/saas_ideas/
------------------------------------------------------------

2. Looking for Micro-SaaS ideas to build or improve
   Author: u/Ielbdev | Subreddit: r/SaaS
   Score: 4 | Comments: 3 | Date: 2025-08-21
   Text: I’m a software engineer building SaaS products. I’m looking for a micro-SaaS idea that solves a focused problem or an existing SaaS that could be improved.

What I’m interested in:

* Tools with payin...
   Link: https://reddit.com/r/SaaS/comments/1mwcvkr/looking_for_microsaas_ideas_to_build_or_improve/
------------------------------------------------------------

3. How do you guys brainstorm ideas to build SaaS?
   Author: u/BlazingBrushes | Subreddit: r/SaaS
   Score: 8 | Comme

In [6]:
# Advanced search examples

def search_and_save_results(queries, filename="reddit_search_results.json"):
    """
    Search for multiple queries and save results to a JSON file
    """
    all_results = {}
    
    for query in queries:
        print(f"Searching for: '{query}'...")
        results = search_reddit_posts(
            query=query,
            subreddit_name="entrepreneur+startups+SaaS+business+sideproject",
            sort="top",
            time_filter="month",
            limit=15
        )
        all_results[query] = results
        print(f"Found {len(results)} posts for '{query}'")
    
    # Save to JSON file
    with open(filename, 'w', encoding='utf-8') as f:
        json.dump(all_results, f, indent=2, default=str)
    
    print(f"\nResults saved to {filename}")
    return all_results

# Example queries for business ideas
idea_queries = [
    "saas ideas",
    "startup ideas 2024",
    "micro saas",
    "business ideas",
    "profitable startup ideas",
    "small business ideas",
    "side project ideas"
]

# Uncomment the line below to run the search
# results = search_and_save_results(idea_queries)

In [7]:
# Utility functions for analyzing search results

def filter_high_quality_posts(posts, min_score=5, min_comments=2):
    """
    Filter posts based on engagement metrics
    """
    return [post for post in posts if post['score'] >= min_score and post['num_comments'] >= min_comments]

def analyze_search_results(posts):
    """
    Analyze search results and provide insights
    """
    if not posts:
        return "No posts to analyze"
    
    total_posts = len(posts)
    avg_score = sum(post['score'] for post in posts) / total_posts
    avg_comments = sum(post['num_comments'] for post in posts) / total_posts
    
    # Top subreddits
    subreddit_counts = {}
    for post in posts:
        subreddit = post['subreddit']
        subreddit_counts[subreddit] = subreddit_counts.get(subreddit, 0) + 1
    
    top_subreddits = sorted(subreddit_counts.items(), key=lambda x: x[1], reverse=True)[:5]
    
    analysis = f"""
    📊 SEARCH RESULTS ANALYSIS
    {'='*50}
    Total posts found: {total_posts}
    Average score: {avg_score:.1f}
    Average comments: {avg_comments:.1f}
    
    📍 Top Subreddits:
    """
    
    for subreddit, count in top_subreddits:
        analysis += f"\n    r/{subreddit}: {count} posts"
    
    return analysis

# Example usage functions
def search_specific_subreddit(query, subreddit):
    """
    Search within a specific subreddit
    """
    print(f"Searching r/{subreddit} for '{query}'...")
    return search_reddit_posts(query, subreddit, limit=10)

# Quick search function for common business idea searches
def quick_idea_search(idea_type="saas"):
    """
    Quick search for different types of business ideas
    """
    search_terms = {
        "saas": "saas ideas OR micro saas OR software ideas",
        "ecommerce": "ecommerce ideas OR online store ideas",
        "service": "service business ideas OR consulting ideas",
        "app": "app ideas OR mobile app startup",
        "ai": "ai startup ideas OR machine learning business"
    }
    
    query = search_terms.get(idea_type.lower(), idea_type)
    return search_reddit_posts(query, "entrepreneur+startups+business", limit=20)

In [8]:
# Analyze the SaaS search results
print(analyze_search_results(saas_posts))

# Filter for high-quality posts
high_quality_posts = filter_high_quality_posts(saas_posts, min_score=8, min_comments=20)
print(f"\n🎯 HIGH QUALITY POSTS (Score ≥8, Comments ≥20): {len(high_quality_posts)}")

for i, post in enumerate(high_quality_posts, 1):
    print(f"\n{i}. {post['title']}")
    print(f"   Score: {post['score']} | Comments: {post['num_comments']}")
    print(f"   Link: {post['permalink']}")

# Example of searching a specific subreddit
print("\n" + "="*80)
print("🔍 SEARCHING SPECIFIC SUBREDDIT: r/entrepreneur")
entrepreneur_posts = search_specific_subreddit("startup ideas", "entrepreneur")
print(f"Found {len(entrepreneur_posts)} posts in r/entrepreneur")


    📊 SEARCH RESULTS ANALYSIS
    Total posts found: 10
    Average score: 7.9
    Average comments: 24.0

    📍 Top Subreddits:
    
    r/SaaS: 9 posts
    r/Entrepreneur: 1 posts

🎯 HIGH QUALITY POSTS (Score ≥8, Comments ≥20): 5

1. SaaS Ideas?
   Score: 11 | Comments: 31
   Link: https://reddit.com/r/SaaS/comments/1n5chir/saas_ideas/

2. How do you guys brainstorm ideas to build SaaS?
   Score: 8 | Comments: 28
   Link: https://reddit.com/r/SaaS/comments/1namong/how_do_you_guys_brainstorm_ideas_to_build_saas/

3. How to Validate Your SaaS Idea Before Launching
   Score: 23 | Comments: 37
   Link: https://reddit.com/r/SaaS/comments/1nd6nac/how_to_validate_your_saas_idea_before_launching/

4. I'm getting fired so I'm gonna focus on building my SaaS idea
   Score: 10 | Comments: 22
   Link: https://reddit.com/r/SaaS/comments/1n8zef9/im_getting_fired_so_im_gonna_focus_on_building_my/

5. I'm a developer with a SaaS idea. How do I get my first customers before I start building?
   Scor